> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# Immutable Source-Aware Fold Builder

Notebook ini membuat pembagian evaluasi tetap untuk seluruh eksperimen revisi:
- **5 outer folds** dengan stratifikasi gabungan `source_dataset × class_label`;
- **4 inner folds** untuk nested cross-validation di dalam setiap outer training set;
- definisi **Leave-One-Dataset-Out (LODO)**;
- checksum SHA-256 untuk memastikan assignment tidak berubah;
- tabel validasi, grafik distribusi, dan PDF audit.

Tidak ada patient grouping buatan. Dataset harus sudah menjadi 594 sampel setelah exact-duplicate exclusion.

**Ubah hanya bagian CONFIGURATION sebelum menjalankan.**

In [ ]:
%pip install pandas numpy matplotlib scikit-learn

In [ ]:
# Immutable Source-Aware Fold Builder
# Purpose:
# - Build fixed, reproducible outer folds for all revised experiments
# - Preserve the joint Source × Class distribution across folds
# - Build immutable inner folds for nested cross-validation
# - Build leave-one-dataset-out (LODO) split definitions
# - Produce CSV assignments, checksums, figures, and a PDF audit report
#
# Edit only the CONFIGURATION section, then run.

from __future__ import annotations

from pathlib import Path
import hashlib
import json
import platform
import sys
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.model_selection import StratifiedKFold
import sklearn


# ============================================================
# CONFIGURATION — EDIT ONLY THIS SECTION
# ============================================================

MANIFEST_PATH = Path(
    r"Dataset Single Source Fixed/"
    r"revised_594_samples_manifest.csv"
)

OUTPUT_DIR = Path(
    r"Dataset Single Source Fixed/immutable_fold_builder"
)

EXPECTED_TOTAL = 594

SOURCE_ORDER = ["ADAM", "FIVES", "ODIR", "RFMID"]
CLASS_ORDER = ["NORMAL", "AMD_DRY", "AMD_WET"]

OUTER_N_SPLITS = 5
INNER_N_SPLITS = 4

OUTER_RANDOM_SEED = 20260717
INNER_RANDOM_SEED_BASE = 20261717

# Joint stratification preserves both repository and class composition.
STRATIFICATION_COLUMN = "source_class_stratum"


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_dataframe(
    dataframe: pd.DataFrame,
    sort_columns: list[str],
) -> str:
    stable = (
        dataframe.sort_values(sort_columns)
        .reset_index(drop=True)
        .to_csv(index=False, lineterminator="\n")
        .encode("utf-8")
    )

    return hashlib.sha256(stable).hexdigest()


def load_manifest() -> pd.DataFrame:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Manifest not found:\n{MANIFEST_PATH}"
        )

    df = pd.read_csv(MANIFEST_PATH)

    required = {
        "sample_id",
        "source_dataset",
        "class_label",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    if len(df) != EXPECTED_TOTAL:
        raise ValueError(
            f"Expected {EXPECTED_TOTAL} rows, found {len(df)}."
        )

    if df["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample_id detected.")

    unexpected_sources = (
        set(df["source_dataset"].dropna().unique())
        - set(SOURCE_ORDER)
    )

    unexpected_classes = (
        set(df["class_label"].dropna().unique())
        - set(CLASS_ORDER)
    )

    if unexpected_sources:
        raise ValueError(
            f"Unexpected source labels: {sorted(unexpected_sources)}"
        )

    if unexpected_classes:
        raise ValueError(
            f"Unexpected class labels: {sorted(unexpected_classes)}"
        )

    df = df.copy()

    df[STRATIFICATION_COLUMN] = (
        df["source_dataset"].astype(str)
        + "__"
        + df["class_label"].astype(str)
    )

    stratum_counts = df[STRATIFICATION_COLUMN].value_counts()

    too_small = stratum_counts.loc[
        stratum_counts < OUTER_N_SPLITS
    ]

    if not too_small.empty:
        raise ValueError(
            "The following Source × Class strata contain fewer samples "
            f"than OUTER_N_SPLITS={OUTER_N_SPLITS}:\n{too_small}"
        )

    print(f"Manifest rows: {len(df)}")
    print("sample_id uniqueness: PASS")
    print("Source labels: PASS")
    print("Class labels: PASS")
    print("Joint Source × Class strata: PASS")

    return df


# ============================================================
# OUTER FOLD BUILDER
# ============================================================

def build_outer_assignments(
    df: pd.DataFrame,
) -> pd.DataFrame:
    splitter = StratifiedKFold(
        n_splits=OUTER_N_SPLITS,
        shuffle=True,
        random_state=OUTER_RANDOM_SEED,
    )

    assignments = df[
        [
            "sample_id",
            "source_dataset",
            "class_label",
            STRATIFICATION_COLUMN,
        ]
    ].copy()

    assignments["outer_fold"] = pd.NA

    x_dummy = np.zeros(len(df))
    y = df[STRATIFICATION_COLUMN].to_numpy()

    for fold_number, (_, validation_indices) in enumerate(
        splitter.split(x_dummy, y),
        start=1,
    ):
        assignments.loc[
            assignments.index[validation_indices],
            "outer_fold",
        ] = fold_number

    assignments["outer_fold"] = assignments[
        "outer_fold"
    ].astype(int)

    if assignments["outer_fold"].isna().any():
        raise RuntimeError("Some samples were not assigned to an outer fold.")

    if assignments["sample_id"].duplicated().any():
        raise RuntimeError("Duplicate sample_id in outer assignments.")

    return assignments.sort_values(
        "sample_id"
    ).reset_index(drop=True)


# ============================================================
# INNER FOLD BUILDER FOR NESTED CV
# ============================================================

def build_nested_assignments(
    outer_assignments: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict] = []

    for outer_fold in range(1, OUTER_N_SPLITS + 1):
        outer_train = outer_assignments.loc[
            outer_assignments["outer_fold"] != outer_fold
        ].copy()

        outer_validation = outer_assignments.loc[
            outer_assignments["outer_fold"] == outer_fold
        ].copy()

        train_stratum_counts = outer_train[
            STRATIFICATION_COLUMN
        ].value_counts()

        too_small = train_stratum_counts.loc[
            train_stratum_counts < INNER_N_SPLITS
        ]

        if not too_small.empty:
            raise ValueError(
                f"Outer fold {outer_fold}: some training strata contain "
                f"fewer than INNER_N_SPLITS={INNER_N_SPLITS}:\n{too_small}"
            )

        inner_splitter = StratifiedKFold(
            n_splits=INNER_N_SPLITS,
            shuffle=True,
            random_state=INNER_RANDOM_SEED_BASE + outer_fold,
        )

        x_dummy = np.zeros(len(outer_train))
        y = outer_train[STRATIFICATION_COLUMN].to_numpy()

        inner_fold_map: dict[str, int] = {}

        for inner_fold, (_, inner_validation_indices) in enumerate(
            inner_splitter.split(x_dummy, y),
            start=1,
        ):
            selected = outer_train.iloc[
                inner_validation_indices
            ]

            for sample_id in selected["sample_id"]:
                inner_fold_map[sample_id] = inner_fold

        if len(inner_fold_map) != len(outer_train):
            raise RuntimeError(
                f"Outer fold {outer_fold}: incomplete inner assignment."
            )

        for row in outer_train.itertuples(index=False):
            rows.append(
                {
                    "sample_id": row.sample_id,
                    "source_dataset": row.source_dataset,
                    "class_label": row.class_label,
                    STRATIFICATION_COLUMN: getattr(
                        row,
                        STRATIFICATION_COLUMN,
                    ),
                    "outer_fold": outer_fold,
                    "outer_role": "train",
                    "inner_fold": inner_fold_map[row.sample_id],
                    "inner_role_definition": (
                        "For a selected inner_fold k, samples with "
                        "inner_fold=k are inner validation; all others "
                        "are inner training."
                    ),
                }
            )

        for row in outer_validation.itertuples(index=False):
            rows.append(
                {
                    "sample_id": row.sample_id,
                    "source_dataset": row.source_dataset,
                    "class_label": row.class_label,
                    STRATIFICATION_COLUMN: getattr(
                        row,
                        STRATIFICATION_COLUMN,
                    ),
                    "outer_fold": outer_fold,
                    "outer_role": "validation",
                    "inner_fold": pd.NA,
                    "inner_role_definition": (
                        "Outer validation sample; never used in inner tuning."
                    ),
                }
            )

    nested = pd.DataFrame(rows)

    nested["inner_fold"] = nested[
        "inner_fold"
    ].astype("Int64")

    return nested.sort_values(
        ["outer_fold", "outer_role", "sample_id"]
    ).reset_index(drop=True)


# ============================================================
# LODO SPLIT DEFINITIONS
# ============================================================

def build_lodo_assignments(
    df: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict] = []

    for held_out_source in SOURCE_ORDER:
        for row in df.itertuples(index=False):
            role = (
                "test"
                if row.source_dataset == held_out_source
                else "train"
            )

            rows.append(
                {
                    "sample_id": row.sample_id,
                    "source_dataset": row.source_dataset,
                    "class_label": row.class_label,
                    STRATIFICATION_COLUMN: getattr(
                        row,
                        STRATIFICATION_COLUMN,
                    ),
                    "held_out_source": held_out_source,
                    "lodo_role": role,
                }
            )

    return pd.DataFrame(rows).sort_values(
        ["held_out_source", "lodo_role", "sample_id"]
    ).reset_index(drop=True)


# ============================================================
# VALIDATION TABLES
# ============================================================

def outer_fold_size_table(
    assignments: pd.DataFrame,
) -> pd.DataFrame:
    return (
        assignments.groupby("outer_fold")
        .size()
        .rename("validation_count")
        .reset_index()
    )


def outer_fold_class_table(
    assignments: pd.DataFrame,
) -> pd.DataFrame:
    table = pd.crosstab(
        assignments["outer_fold"],
        assignments["class_label"],
    )

    return table.reindex(
        index=range(1, OUTER_N_SPLITS + 1),
        columns=CLASS_ORDER,
        fill_value=0,
    )


def outer_fold_source_table(
    assignments: pd.DataFrame,
) -> pd.DataFrame:
    table = pd.crosstab(
        assignments["outer_fold"],
        assignments["source_dataset"],
    )

    return table.reindex(
        index=range(1, OUTER_N_SPLITS + 1),
        columns=SOURCE_ORDER,
        fill_value=0,
    )


def outer_fold_source_class_table(
    assignments: pd.DataFrame,
) -> pd.DataFrame:
    table = pd.crosstab(
        assignments["outer_fold"],
        assignments[STRATIFICATION_COLUMN],
    )

    strata_order = [
        f"{source}__{class_label}"
        for source in SOURCE_ORDER
        for class_label in CLASS_ORDER
        if (
            assignments[STRATIFICATION_COLUMN]
            == f"{source}__{class_label}"
        ).any()
    ]

    return table.reindex(
        index=range(1, OUTER_N_SPLITS + 1),
        columns=strata_order,
        fill_value=0,
    )


def fold_balance_summary(
    assignments: pd.DataFrame,
) -> pd.DataFrame:
    class_table = outer_fold_class_table(assignments)
    source_table = outer_fold_source_table(assignments)
    size_table = outer_fold_size_table(assignments)

    rows = []

    for fold in range(1, OUTER_N_SPLITS + 1):
        class_counts = class_table.loc[fold]
        source_counts = source_table.loc[fold]
        fold_size = int(
            size_table.loc[
                size_table["outer_fold"] == fold,
                "validation_count",
            ].iloc[0]
        )

        rows.append(
            {
                "outer_fold": fold,
                "validation_count": fold_size,
                "normal_count": int(class_counts["NORMAL"]),
                "dry_count": int(class_counts["AMD_DRY"]),
                "wet_count": int(class_counts["AMD_WET"]),
                "adam_count": int(source_counts["ADAM"]),
                "fives_count": int(source_counts["FIVES"]),
                "odir_count": int(source_counts["ODIR"]),
                "rfmid_count": int(source_counts["RFMID"]),
            }
        )

    return pd.DataFrame(rows)


def validate_outer_assignments(
    df: pd.DataFrame,
    assignments: pd.DataFrame,
) -> dict:
    checks = {}

    checks["total_rows_match"] = len(assignments) == len(df)
    checks["unique_sample_ids"] = (
        assignments["sample_id"].nunique() == len(df)
    )
    checks["all_folds_present"] = (
        set(assignments["outer_fold"].unique())
        == set(range(1, OUTER_N_SPLITS + 1))
    )
    checks["no_missing_fold"] = (
        assignments["outer_fold"].isna().sum() == 0
    )

    stratum_table = outer_fold_source_class_table(assignments)

    # For joint stratification, counts for each stratum should differ
    # by at most one across folds.
    checks["stratum_max_difference_le_1"] = bool(
        (stratum_table.max(axis=0) - stratum_table.min(axis=0))
        .le(1)
        .all()
    )

    if not all(checks.values()):
        failed = [
            name for name, passed in checks.items()
            if not passed
        ]

        raise RuntimeError(
            f"Outer assignment validation failed: {failed}"
        )

    return checks


def validate_nested_assignments(
    outer_assignments: pd.DataFrame,
    nested_assignments: pd.DataFrame,
) -> dict:
    checks = {}

    expected_rows = (
        len(outer_assignments) * OUTER_N_SPLITS
    )

    checks["expected_total_rows"] = (
        len(nested_assignments) == expected_rows
    )

    for outer_fold in range(1, OUTER_N_SPLITS + 1):
        subset = nested_assignments.loc[
            nested_assignments["outer_fold"] == outer_fold
        ]

        validation_samples = subset.loc[
            subset["outer_role"] == "validation",
            "sample_id",
        ]

        expected_validation = outer_assignments.loc[
            outer_assignments["outer_fold"] == outer_fold,
            "sample_id",
        ]

        if set(validation_samples) != set(expected_validation):
            checks[
                f"outer_validation_match_fold_{outer_fold}"
            ] = False
        else:
            checks[
                f"outer_validation_match_fold_{outer_fold}"
            ] = True

        train_subset = subset.loc[
            subset["outer_role"] == "train"
        ]

        checks[
            f"all_inner_folds_present_outer_{outer_fold}"
        ] = (
            set(train_subset["inner_fold"].dropna().astype(int).unique())
            == set(range(1, INNER_N_SPLITS + 1))
        )

        checks[
            f"outer_validation_has_no_inner_fold_{outer_fold}"
        ] = validation_samples.shape[0] == subset.loc[
            subset["outer_role"] == "validation",
            "inner_fold",
        ].isna().sum()

    if not all(checks.values()):
        failed = [
            name for name, passed in checks.items()
            if not passed
        ]

        raise RuntimeError(
            f"Nested assignment validation failed: {failed}"
        )

    return checks


# ============================================================
# FIGURES
# ============================================================

def save_fold_class_figure(
    class_table: pd.DataFrame,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))

    bottom = np.zeros(len(class_table))

    for class_label in CLASS_ORDER:
        values = class_table[class_label].to_numpy()

        ax.bar(
            class_table.index.astype(str),
            values,
            bottom=bottom,
            label=class_label,
        )

        bottom += values

    ax.set_title("Outer-Fold Validation Class Distribution")
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Number of validation images")
    ax.legend(title="Class")

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_fold_source_figure(
    source_table: pd.DataFrame,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))

    bottom = np.zeros(len(source_table))

    for source in SOURCE_ORDER:
        values = source_table[source].to_numpy()

        ax.bar(
            source_table.index.astype(str),
            values,
            bottom=bottom,
            label=source,
        )

        bottom += values

    ax.set_title("Outer-Fold Validation Source Distribution")
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Number of validation images")
    ax.legend(title="Source")

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_fold_stratum_heatmap(
    stratum_table: pd.DataFrame,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(14, 6))

    image = ax.imshow(
        stratum_table.to_numpy(),
        aspect="auto",
    )

    ax.set_xticks(range(len(stratum_table.columns)))
    ax.set_xticklabels(
        stratum_table.columns,
        rotation=45,
        ha="right",
    )

    ax.set_yticks(range(len(stratum_table.index)))
    ax.set_yticklabels(stratum_table.index)

    ax.set_title("Outer-Fold Source × Class Stratum Counts")
    ax.set_xlabel("Source × class stratum")
    ax.set_ylabel("Outer fold")

    for row in range(stratum_table.shape[0]):
        for col in range(stratum_table.shape[1]):
            ax.text(
                col,
                row,
                str(int(stratum_table.iloc[row, col])),
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Validation samples",
    )

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# PDF REPORT
# ============================================================

def add_title_page(
    pdf: PdfPages,
    manifest_hash: str,
    assignment_hash: str,
) -> None:
    fig = plt.figure(figsize=(11.69, 8.27))

    fig.suptitle(
        "Immutable Source-Aware Fold Assignment Report",
        fontsize=20,
        fontweight="bold",
        y=0.94,
    )

    text = (
        f"Dataset: {EXPECTED_TOTAL} duplicate-controlled images\n"
        f"Outer folds: {OUTER_N_SPLITS}\n"
        f"Inner folds per outer training set: {INNER_N_SPLITS}\n"
        f"Outer random seed: {OUTER_RANDOM_SEED}\n"
        f"Inner seed base: {INNER_RANDOM_SEED_BASE}\n"
        f"Stratification unit: Source × Class joint stratum\n\n"
        "Rationale:\n"
        "Patient identifiers were not consistently available across "
        "repositories. Therefore, unverifiable patient groups were not "
        "constructed. Instead, all confirmed exact duplicates were removed "
        "before fold construction, and the immutable outer folds were "
        "stratified by the joint repository–class label to preserve both "
        "source and class composition.\n\n"
        f"Input manifest SHA-256:\n{manifest_hash}\n\n"
        f"Outer assignment SHA-256:\n{assignment_hash}"
    )

    fig.text(
        0.08,
        0.80,
        text,
        va="top",
        fontsize=11.5,
        linespacing=1.45,
    )

    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_table_page(
    pdf: PdfPages,
    title: str,
    dataframe: pd.DataFrame,
    fontsize: int = 9,
) -> None:
    fig = plt.figure(figsize=(11.69, 8.27))

    fig.suptitle(
        title,
        fontsize=17,
        fontweight="bold",
        y=0.96,
    )

    ax = fig.add_axes([0.04, 0.10, 0.92, 0.78])
    ax.axis("off")

    display_df = dataframe.copy()

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.5)

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_figure_page(
    pdf: PdfPages,
    title: str,
    image_path: Path,
) -> None:
    image = plt.imread(image_path)

    fig = plt.figure(figsize=(11.69, 8.27))

    fig.suptitle(
        title,
        fontsize=17,
        fontweight="bold",
        y=0.96,
    )

    ax = fig.add_axes([0.05, 0.07, 0.90, 0.82])
    ax.imshow(image)
    ax.axis("off")

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_protocol_page(
    pdf: PdfPages,
) -> None:
    text = (
        "How to use the assignments\n\n"
        "Outer cross-validation:\n"
        "For outer fold k, samples with outer_fold=k are the untouched "
        "outer validation set. All remaining samples form the outer "
        "training set.\n\n"
        "Nested hyperparameter tuning:\n"
        "Within outer fold k, use only rows with outer_role='train'. "
        "For inner fold j, samples with inner_fold=j are inner validation; "
        "all other outer-training samples are inner training. The outer "
        "validation samples have no inner-fold assignment and must never "
        "influence model selection.\n\n"
        "Leave-one-dataset-out evaluation:\n"
        "For held_out_source=s, samples from s are test data and all "
        "remaining repositories are training data. Because ADAM contains "
        "no NORMAL images, held-out ADAM results must explicitly report "
        "the available classes and must not be interpreted as a directly "
        "comparable three-class test.\n\n"
        "Immutability rule:\n"
        "Do not regenerate or change these assignments after experiments "
        "begin. All models, ablations, baselines, timing studies, and paired "
        "statistical comparisons must use these same assignments."
    )

    fig = plt.figure(figsize=(11.69, 8.27))

    fig.suptitle(
        "Evaluation Protocol and Immutability Rules",
        fontsize=17,
        fontweight="bold",
        y=0.96,
    )

    fig.text(
        0.07,
        0.87,
        text,
        va="top",
        fontsize=11,
        linespacing=1.4,
    )

    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)



# ============================================================
# JSON SERIALIZATION HELPER
# ============================================================

def make_json_serializable(value):
    """
    Convert NumPy/Pandas scalar objects recursively into native
    Python objects that json.dump() can serialize.
    """
    if isinstance(value, dict):
        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            make_json_serializable(item)
            for item in value
        ]

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if value is pd.NA:
        return None

    if isinstance(value, Path):
        return str(value)

    return value


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_manifest()

    outer_assignments = build_outer_assignments(df)

    outer_checks = validate_outer_assignments(
        df,
        outer_assignments,
    )

    nested_assignments = build_nested_assignments(
        outer_assignments
    )

    nested_checks = validate_nested_assignments(
        outer_assignments,
        nested_assignments,
    )

    lodo_assignments = build_lodo_assignments(df)

    fold_sizes = outer_fold_size_table(outer_assignments)
    class_table = outer_fold_class_table(outer_assignments)
    source_table = outer_fold_source_table(outer_assignments)
    stratum_table = outer_fold_source_class_table(outer_assignments)
    balance_summary = fold_balance_summary(outer_assignments)

    # Save primary assignments.
    outer_path = OUTPUT_DIR / "outer_fold_assignments.csv"
    nested_path = OUTPUT_DIR / "nested_cv_assignments.csv"
    lodo_path = OUTPUT_DIR / "lodo_split_assignments.csv"

    outer_assignments.to_csv(
        outer_path,
        index=False,
        encoding="utf-8-sig",
    )

    nested_assignments.to_csv(
        nested_path,
        index=False,
        encoding="utf-8-sig",
    )

    lodo_assignments.to_csv(
        lodo_path,
        index=False,
        encoding="utf-8-sig",
    )

    # Save validation tables.
    fold_sizes.to_csv(
        OUTPUT_DIR / "outer_fold_sizes.csv",
        index=False,
        encoding="utf-8-sig",
    )

    class_table.to_csv(
        OUTPUT_DIR / "outer_fold_class_distribution.csv",
        encoding="utf-8-sig",
    )

    source_table.to_csv(
        OUTPUT_DIR / "outer_fold_source_distribution.csv",
        encoding="utf-8-sig",
    )

    stratum_table.to_csv(
        OUTPUT_DIR / "outer_fold_source_class_distribution.csv",
        encoding="utf-8-sig",
    )

    balance_summary.to_csv(
        OUTPUT_DIR / "outer_fold_balance_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Checksums.
    manifest_hash = sha256_file(MANIFEST_PATH)
    outer_file_hash = sha256_file(outer_path)
    nested_file_hash = sha256_file(nested_path)
    lodo_file_hash = sha256_file(lodo_path)

    outer_dataframe_hash = sha256_dataframe(
        outer_assignments,
        ["sample_id"],
    )

    checksum_table = pd.DataFrame(
        [
            {
                "artifact": "input_manifest",
                "path": str(MANIFEST_PATH),
                "sha256": manifest_hash,
            },
            {
                "artifact": "outer_fold_assignments",
                "path": str(outer_path),
                "sha256": outer_file_hash,
            },
            {
                "artifact": "nested_cv_assignments",
                "path": str(nested_path),
                "sha256": nested_file_hash,
            },
            {
                "artifact": "lodo_split_assignments",
                "path": str(lodo_path),
                "sha256": lodo_file_hash,
            },
            {
                "artifact": "outer_assignments_canonical_dataframe",
                "path": "canonical CSV representation",
                "sha256": outer_dataframe_hash,
            },
        ]
    )

    checksum_table.to_csv(
        OUTPUT_DIR / "assignment_checksums.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Reproducibility metadata.
    metadata = {
        "expected_total": EXPECTED_TOTAL,
        "outer_n_splits": OUTER_N_SPLITS,
        "inner_n_splits": INNER_N_SPLITS,
        "outer_random_seed": OUTER_RANDOM_SEED,
        "inner_random_seed_base": INNER_RANDOM_SEED_BASE,
        "stratification_column": STRATIFICATION_COLUMN,
        "source_order": SOURCE_ORDER,
        "class_order": CLASS_ORDER,
        "python_version": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "scikit_learn_version": sklearn.__version__,
        "outer_validation_checks": outer_checks,
        "nested_validation_checks": nested_checks,
        "input_manifest_sha256": manifest_hash,
        "outer_assignment_sha256": outer_file_hash,
        "nested_assignment_sha256": nested_file_hash,
        "lodo_assignment_sha256": lodo_file_hash,
    }

    metadata_serializable = make_json_serializable(metadata)

    with (
        OUTPUT_DIR / "fold_builder_metadata.json"
    ).open("w", encoding="utf-8") as file:
        json.dump(
            metadata_serializable,
            file,
            indent=2,
            ensure_ascii=False,
        )

    # Figures.
    class_figure = OUTPUT_DIR / "outer_fold_class_distribution.png"
    source_figure = OUTPUT_DIR / "outer_fold_source_distribution.png"
    stratum_figure = OUTPUT_DIR / "outer_fold_source_class_heatmap.png"

    save_fold_class_figure(
        class_table,
        class_figure,
    )

    save_fold_source_figure(
        source_table,
        source_figure,
    )

    save_fold_stratum_heatmap(
        stratum_table,
        stratum_figure,
    )

    # PDF audit report.
    pdf_path = (
        OUTPUT_DIR
        / "immutable_source_aware_fold_assignment_report.pdf"
    )

    with PdfPages(pdf_path) as pdf:
        add_title_page(
            pdf,
            manifest_hash,
            outer_file_hash,
        )

        add_table_page(
            pdf,
            "Outer-Fold Validation Sizes",
            fold_sizes,
        )

        add_table_page(
            pdf,
            "Outer-Fold Class Distribution",
            class_table.reset_index(),
        )

        add_table_page(
            pdf,
            "Outer-Fold Source Distribution",
            source_table.reset_index(),
        )

        add_table_page(
            pdf,
            "Outer-Fold Balance Summary",
            balance_summary,
            fontsize=8,
        )

        add_figure_page(
            pdf,
            "Outer-Fold Validation Class Distribution",
            class_figure,
        )

        add_figure_page(
            pdf,
            "Outer-Fold Validation Source Distribution",
            source_figure,
        )

        add_figure_page(
            pdf,
            "Outer-Fold Source × Class Strata",
            stratum_figure,
        )

        add_table_page(
            pdf,
            "Assignment Checksums",
            checksum_table,
            fontsize=7,
        )

        add_protocol_page(pdf)

    print("\n=== IMMUTABLE SOURCE-AWARE FOLD BUILDER ===")

    print("\nOuter-fold sizes:")
    print(fold_sizes.to_string(index=False))

    print("\nOuter-fold class distribution:")
    print(class_table)

    print("\nOuter-fold source distribution:")
    print(source_table)

    print("\nValidation checks:")
    print("Outer:", outer_checks)
    print("Nested:", nested_checks)

    print("\nOutputs saved to:")
    print(OUTPUT_DIR)

    print("\nPrimary assignment files:")
    print(outer_path)
    print(nested_path)
    print(lodo_path)

    print("\nPDF report:")
    print(pdf_path)


if __name__ == "__main__":
    main()
